## Useful libraries

In [57]:
import pandas as pd
import requests
import os
from geopy.geocoders import Nominatim
from dotenv import load_dotenv

In [58]:
load_dotenv()

True

###### **Challenge:** Arrivals information
Using what you have been shown above, plus the skills you've learnt in the last couple of days:
1. In `AeroDataBox API` use the `Flight API` > `FIDS/Schedules: Airport departures and arrivals (by time range)` section
2. Fill out the parameters in the middle third and then copy the `python: requests` code from the right hand third
3. Explore the data you get back. What would be useful in your DataFrame and what can be excluded? Remember Gans wants to know about when people are arriving in the city
4. Make a DataFrame from the information you see as important
5. Condense everything you did above into a function that can take a list of ICAO codes as an input, and as an output gives you a DataFrame with the information for *tomorrows arrivals*

## 1. Create variable with API_key

In [ ]:
API_key = os.getenv("AeroDataBoxApi")
if API_key is None:
    print("Error: API_key not found in .env file!")
else:
    print(f"Key loaded successfully: {API_key[:2]}...") # prints first 5 characters

Key loaded successfully: 68729...


## 2. Get info airports in Berlin, Hamburg and Munich

2.1. Let's say in the beginning we would like to get info in Berlin

In [61]:
url = "https://aerodatabox.p.rapidapi.com/airports/search/location/52.31/13.24/km/50/16"

querystring = {"withFlightInfoOnly":"true"}

headers = {
	"X-RapidAPI-Key": API_key.strip(),
	"X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
}

response = requests.request("GET", url, headers=headers, params=querystring)

print(response.text)

{"searchBy":{"lat":52.31,"lon":13.24},"count":2,"items":[{"icao":"EDDB","iata":"BER","name":"Berlin Brandenburg","shortName":"Brandenburg","municipalityName":"Berlin","location":{"lat":52.35139,"lon":13.493889},"countryCode":"DE","timeZone":"Europe/Berlin"},{"icao":"EDDT","iata":"TXL","name":"Berlin -Tegel","shortName":"-Tegel","municipalityName":"Berlin","location":{"lat":52.5597,"lon":13.287699},"countryCode":"DE","timeZone":"Europe/Berlin"}]}


2.2. Create for loop to get info in all 3 cities

In [62]:
# setup Geocoder (no API key needed for Nominatim)
geolocator = Nominatim(user_agent="my_airport_search_app")
# define cities:
cities = ['Berlin', 'Hamburg', 'Munich']
all_airports = []
for city in cities:
    # get coordinates for the city
    location = geolocator.geocode(city)
    if location:
        lat, lon = location.latitude, location.longitude
        print(f"Searching airports near {city}({lat},{lon})")
        # constract AeroBox url format
        url = f"https://aerodatabox.p.rapidapi.com/airports/search/location/{lat}/{lon}/km/50/16"
        querystring = {"withFlightInfoOnly":"true"}
        headers = {
	"X-RapidAPI-Key": API_key.strip(),
	"X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
}
        response = requests.request("GET", url, headers=headers, params=querystring)
        if response.status_code == 200:
            data = response.json().get('items', [])
            all_airports.extend(data)
            print(response.json())
        else:
            print(f"Error for {city}: {response.json().get('message')}")
    else:
        print(f"Could not find coordinates for {city}")

airports_df = pd.json_normalize(all_airports)
airports_df

Searching airports near Berlin(52.5173885,13.3951309)
{'searchBy': {'lat': 52.517387, 'lon': 13.395131}, 'count': 2, 'items': [{'icao': 'EDDT', 'iata': 'TXL', 'name': 'Berlin -Tegel', 'shortName': '-Tegel', 'municipalityName': 'Berlin', 'location': {'lat': 52.5597, 'lon': 13.287699}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}, {'icao': 'EDDB', 'iata': 'BER', 'name': 'Berlin Brandenburg', 'shortName': 'Brandenburg', 'municipalityName': 'Berlin', 'location': {'lat': 52.35139, 'lon': 13.493889}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}]}
Searching airports near Hamburg(53.550341,10.000654)
{'searchBy': {'lat': 53.550343, 'lon': 10.000654}, 'count': 1, 'items': [{'icao': 'EDDH', 'iata': 'HAM', 'name': 'Hamburg', 'shortName': 'Hamburg', 'municipalityName': 'Hamburg', 'location': {'lat': 53.6304, 'lon': 9.988229}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}]}
Searching airports near Munich(48.1371079,11.5753822)
{'searchBy': {'lat': 48.137108, 'lon': 11.575382}, 'count'

,icao,iata,name,shortName,municipalityName,countryCode,timeZone,location.lat,location.lon
0,EDDT,TXL,Berlin -Tegel,-Tegel,Berlin,DE,Europe/Berlin,52.55970,13.287699
1,EDDB,BER,Berlin Brandenburg,Brandenburg,Berlin,DE,Europe/Berlin,52.35139,13.493889
2,EDDH,HAM,Hamburg,Hamburg,Hamburg,DE,Europe/Berlin,53.63040,9.988229
3,EDDM,MUC,Munich,Munich,Munich,DE,Europe/Berlin,48.35380,11.786100


Actually, both rows for Berlin are technically "correct" in a database sense, but Tegel (TXL) is permanently closed (it shut down in 2020). Brandenburg (BER) is the only active commercial airport in Berlin.

In [63]:
# remove TXL from our DataFrame
clean_df = airports_df[airports_df['iata'] != 'TXL'].copy()
clean_df


,icao,iata,name,shortName,municipalityName,countryCode,timeZone,location.lat,location.lon
1,EDDB,BER,Berlin Brandenburg,Brandenburg,Berlin,DE,Europe/Berlin,52.35139,13.493889
2,EDDH,HAM,Hamburg,Hamburg,Hamburg,DE,Europe/Berlin,53.63040,9.988229
3,EDDM,MUC,Munich,Munich,Munich,DE,Europe/Berlin,48.35380,11.786100


2.3. Rename the columns in our DataFrame:

In [64]:
# define a dictionary mapping the old names to new names
column_mapping = {
    'icao': 'airport_icao',
    'iata': 'iata',
    'name': 'airport_name',
    'shortName': 'short_name',
    'municipalityName': 'city',
    'countryCode': 'country_code',
    'timeZone': 'timezone',
    'location.lat': 'latitude',
    'location.lon': 'longitude'
       }

# apply the rename
main_df = clean_df.rename(columns=column_mapping)

# view the final result
main_df

,airport_icao,iata,airport_name,short_name,city,country_code,timezone,latitude,longitude
1,EDDB,BER,Berlin Brandenburg,Brandenburg,Berlin,DE,Europe/Berlin,52.35139,13.493889
2,EDDH,HAM,Hamburg,Hamburg,Hamburg,DE,Europe/Berlin,53.63040,9.988229
3,EDDM,MUC,Munich,Munich,Munich,DE,Europe/Berlin,48.35380,11.786100


## 3. Retrieving our DataFrame to SQL

In [65]:
# configuration and connection
def connection_string_local():
    schema = "sql_gans"
    host = "127.0.0.1"
    user = "root"
    password = os.getenv("password")
    port = 3306

    if password is None:
        raise ValueError("ERROR: 'password' not found in .env file. Check your filename and path.")

# build connection string only if password exists
    connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
    print("Connection string created successfully.")
    return connection_string

In [66]:
connection_string = connection_string_local()

Connection string created successfully.


In [67]:
print(main_df.columns)

Index(['airport_icao', 'iata', 'airport_name', 'short_name', 'city',
       'country_code', 'timezone', 'latitude', 'longitude'],
      dtype='str')


3.1. Merging our airports DataFrame with `cities` table.

In [68]:
# dataFrame for the 'airports' table
# should include: airport_icao, iata, airport_name, etc.
airports_df = main_df[['airport_icao', 'iata', 'airport_name', 'short_name',
                       'country_code', 'timezone', 'latitude', 'longitude']].drop_duplicates()

# read cities table from MySql
cities_from_sql = pd.read_sql_table('cities', con=connection_string)
# merge cities and airports to get `city_id`
merged_df = main_df.merge(cities_from_sql[['city_id', 'city']],
                              on='city',
                              how='left')

# dataFrame for the 'cities_airports' junction table
# should include: city_id, airport_icao
cities_airports_df = merged_df[['city_id', 'airport_icao']].drop_duplicates()

3.2. Create a function to send data from our DataFrames to MySql 

In [15]:
!pip install cryptography


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Be careful - if you try to run the function below again you will get an error: `SQL Error: Execution failed on sql 'INSERT INTO cities_airports (city_id, airport_icao) VALUES (:city_id, :airport_icao)': (pymysql.err.IntegrityError) (1062, "Duplicate entry 'EDDB' for key 'cities_airports.PRIMARY'")
[SQL: INSERT INTO cities_airports (city_id, airport_icao) VALUES (%(city_id)s, %(airport_icao)s)]
[parameters: [{'city_id': 1, 'airport_icao': 'EDDB'}, {'city_id': 2, 'airport_icao': 'EDDH'}, {'city_id': 3, 'airport_icao': 'EDDM'}]]
(Background on this error at: https://sqlalche.me/e/20/gkpj)`. That's expected since MySQL avoids the duplicates according to the code.

In [69]:
def send_airports_to_sql(airports_df, cities_airports_df, connection_string):
    # define columns for the 'airports' table
    airports_columns = [
        'airport_icao', 'iata', 'airport_name', 'short_name',
        'country_code', 'timezone', 'latitude', 'longitude'
    ]

    # define columns for the 'cities_airports' table
    cities_airports_columns = ['city_id', 'airport_icao']
    print("Current columns in airports_df:", airports_df.columns.tolist())

    try:
         # upload junction first (since DB requires this order)
        data_junction = cities_airports_df[cities_airports_columns].drop_duplicates()
        data_junction.to_sql(name='cities_airports', con=connection_string, if_exists='append', index=False)
        print(f"Success: {len(data_junction)} rows synced to 'cities_airports'.")

        # upload airports second
        data_airports = airports_df[airports_columns].drop_duplicates()
        data_airports.to_sql(name='airports', con=connection_string, if_exists='append', index=False)
        print(f"Success: {len(data_airports)} rows synced to 'airports'.")

    except Exception as e:
        print(f"SQL Error: {e}")

send_airports_to_sql(airports_df, cities_airports_df, connection_string)


Current columns in airports_df: ['airport_icao', 'iata', 'airport_name', 'short_name', 'country_code', 'timezone', 'latitude', 'longitude']
Success: 3 rows synced to 'cities_airports'.
Success: 3 rows synced to 'airports'.


## 4. Get info about arrivals

In [ ]:
#!pip install pytz


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [70]:
datetime.now().date()

datetime.date(2026, 3, 3)

In [71]:
# api_key = userdata.get('aeroBoxDataApi')
API_key = os.getenv("AeroDataBoxApi")

# provide ICAO, date, and time manually to test
icao = "EDDB"
date = datetime.now().date()
time_start = f"{date}T00:00"
time_end = f"{date}T11:59"

# constructing url for API call
url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{time_start}/{time_end}"

querystring = {"withLeg":"true",        # where is flight arriving from
               "direction":"Arrival",
               "withCancelled":"false",
               "withCodeshared":"true",
               "withCargo":"false",
               "withPrivate":"false"}

headers = {
    'x-rapidapi-host': "aerodatabox.p.rapidapi.com",
    'x-rapidapi-key': API_key
    }

# grabbing response
response = requests.request("GET",
                            url,
                            headers = headers,
                            params = querystring)

flights_json = response.json()

flights_json

{'arrivals': [{'departure': {'airport': {'icao': 'KEWR',
     'iata': 'EWR',
     'name': 'Newark',
     'countryCode': 'us',
     'timeZone': 'America/New_York'},
    'scheduledTime': {'utc': '2026-03-02 22:50Z',
     'local': '2026-03-02 17:50-05:00'},
    'revisedTime': {'utc': '2026-03-02 23:13Z',
     'local': '2026-03-02 18:13-05:00'},
    'runwayTime': {'utc': '2026-03-02 23:13Z',
     'local': '2026-03-02 18:13-05:00'},
    'terminal': 'C',
    'quality': ['Basic', 'Live']},
   'arrival': {'scheduledTime': {'utc': '2026-03-03 06:55Z',
     'local': '2026-03-03 07:55+01:00'},
    'revisedTime': {'utc': '2026-03-03 06:27Z',
     'local': '2026-03-03 07:27+01:00'},
    'runwayTime': {'utc': '2026-03-03 06:28Z',
     'local': '2026-03-03 07:28+01:00'},
    'terminal': '1',
    'gate': 'Y17',
    'baggageBelt': 'B2',
    'runway': '25R',
    'quality': ['Basic', 'Live']},
   'number': 'UA 962',
   'callSign': 'UAL962',
   'status': 'Arrived',
   'codeshareStatus': 'IsOperator',
   '

4.1. Exploring json

In [72]:
flights_json.keys()

dict_keys(['arrivals'])

In [73]:
flights_json["arrivals"]

[{'departure': {'airport': {'icao': 'KEWR',
    'iata': 'EWR',
    'name': 'Newark',
    'countryCode': 'us',
    'timeZone': 'America/New_York'},
   'scheduledTime': {'utc': '2026-03-02 22:50Z',
    'local': '2026-03-02 17:50-05:00'},
   'revisedTime': {'utc': '2026-03-02 23:13Z',
    'local': '2026-03-02 18:13-05:00'},
   'runwayTime': {'utc': '2026-03-02 23:13Z',
    'local': '2026-03-02 18:13-05:00'},
   'terminal': 'C',
   'quality': ['Basic', 'Live']},
  'arrival': {'scheduledTime': {'utc': '2026-03-03 06:55Z',
    'local': '2026-03-03 07:55+01:00'},
   'revisedTime': {'utc': '2026-03-03 06:27Z',
    'local': '2026-03-03 07:27+01:00'},
   'runwayTime': {'utc': '2026-03-03 06:28Z',
    'local': '2026-03-03 07:28+01:00'},
   'terminal': '1',
   'gate': 'Y17',
   'baggageBelt': 'B2',
   'runway': '25R',
   'quality': ['Basic', 'Live']},
  'number': 'UA 962',
  'callSign': 'UAL962',
  'status': 'Arrived',
  'codeshareStatus': 'IsOperator',
  'isCargo': False,
  'aircraft': {'reg': 'N

The square brackets at the beginning of `flights_json` indicate that it represents a list-like structure. Since we're dealing with a list, we can iterate through its elements to access and process the data. Let's start by examining the first element in the list.

In [74]:
flights_json["arrivals"][0]

{'departure': {'airport': {'icao': 'KEWR',
   'iata': 'EWR',
   'name': 'Newark',
   'countryCode': 'us',
   'timeZone': 'America/New_York'},
  'scheduledTime': {'utc': '2026-03-02 22:50Z',
   'local': '2026-03-02 17:50-05:00'},
  'revisedTime': {'utc': '2026-03-02 23:13Z',
   'local': '2026-03-02 18:13-05:00'},
  'runwayTime': {'utc': '2026-03-02 23:13Z',
   'local': '2026-03-02 18:13-05:00'},
  'terminal': 'C',
  'quality': ['Basic', 'Live']},
 'arrival': {'scheduledTime': {'utc': '2026-03-03 06:55Z',
   'local': '2026-03-03 07:55+01:00'},
  'revisedTime': {'utc': '2026-03-03 06:27Z',
   'local': '2026-03-03 07:27+01:00'},
  'runwayTime': {'utc': '2026-03-03 06:28Z',
   'local': '2026-03-03 07:28+01:00'},
  'terminal': '1',
  'gate': 'Y17',
  'baggageBelt': 'B2',
  'runway': '25R',
  'quality': ['Basic', 'Live']},
 'number': 'UA 962',
 'callSign': 'UAL962',
 'status': 'Arrived',
 'codeshareStatus': 'IsOperator',
 'isCargo': False,
 'aircraft': {'reg': 'N76065', 'modeS': 'AA44D0', 'mo

In [75]:
flights_json["arrivals"][0].keys()

dict_keys(['departure', 'arrival', 'number', 'callSign', 'status', 'codeshareStatus', 'isCargo', 'aircraft', 'airline'])

Looking at the first element of the json and the available keys, we can select the information we think would be important for our dataframe.
- Departure airport icao
- scheduled arrival time, local
- flight number

4.2. Using for loops create DataFrame

In [76]:
# loop to extract flight data
flight_items = []

# assuming flights_json is the response you received
for item in flights_json["arrivals"]:
    flight_item = {
        "arrival_aeroport_icao": icao,  # arrival airport ICAO (from the loop or API)
        "departure_airport_icao": item["departure"]["airport"].get("icao", None),  # departure airport ICAO
        "flight_number": item.get("number", None),  # flight number
        "airline": item.get("airline", {}).get("name", None),  # airline name (United)
        "arrival_time": item.get("arrival", {}).get("scheduledTime", {}).get("local", None),  # scheduled arrival time
        "arrival_terminal": item.get("arrival", {}).get("terminal", None),  # arrival terminal
        "departure_city": item.get("departure", {}).get("airport", {}).get("name", None),  # departure city (airport name)
        "data_retrieved_on": datetime.now()  # timestamp of when the data was retrieved
    }

    flight_items.append(flight_item)

# convert to DataFrame
flights_df = pd.DataFrame(flight_items)

# convert arrival time to datetime format
flights_df["arrival_time"] = pd.to_datetime(flights_df["arrival_time"])

# display first few rows
flights_df

,arrival_aeroport_icao,departure_airport_icao,flight_number,airline,arrival_time,arrival_terminal,departure_city,data_retrieved_on
0,EDDB,KEWR,UA 962,United,2026-03-03 07:55:00+01:00,1,Newark,2026-03-03 14:46:14.529397
1,EDDB,LOWS,EW 4342,Eurowings,2026-03-03 08:00:00+01:00,1,Salzburg,2026-03-03 14:46:14.529429
2,EDDB,EFHK,AY 1431,Finnair,2026-03-03 07:55:00+01:00,1,Helsinki,2026-03-03 14:46:14.529448
3,EDDB,EVRA,BT 211,airBaltic,2026-03-03 07:55:00+01:00,1,Riga,2026-03-03 14:46:14.529458
4,EDDB,EVRA,JU 7659,JU,2026-03-03 07:55:00+01:00,1,Riga,2026-03-03 14:46:14.529471
...,...,...,...,...,...,...,...,...
104,EDDB,EKCH,D8 3302,Norwegian Air Sweden,2026-03-03 12:00:00+01:00,2,Copenhagen,2026-03-03 14:46:14.533585
105,EDDB,LICC,FR 1577,Ryanair,2026-03-03 12:00:00+01:00,2,Catania,2026-03-03 14:46:14.533592
106,EDDB,EDDF,VL 180,Lufthansa City,2026-03-03 11:50:00+01:00,1,Frankfurt-am-Main,2026-03-03 14:46:14.533600
107,EDDB,EFHK,AY 1433,Finnair,2026-03-03 12:15:00+01:00,1,Helsinki,2026-03-03 14:46:14.533607


Let's get rid of the `+01:00` from `arrival_time`.

In [77]:
# slice and convert to actual datetime objects
flights_df["arrival_time"] = flights_df["arrival_time"].dt.tz_localize(None)
flights_df.head()

,arrival_aeroport_icao,departure_airport_icao,flight_number,airline,arrival_time,arrival_terminal,departure_city,data_retrieved_on
0,EDDB,KEWR,UA 962,United,2026-03-03 07:55:00,1,Newark,2026-03-03 14:46:14.529397
1,EDDB,LOWS,EW 4342,Eurowings,2026-03-03 08:00:00,1,Salzburg,2026-03-03 14:46:14.529429
2,EDDB,EFHK,AY 1431,Finnair,2026-03-03 07:55:00,1,Helsinki,2026-03-03 14:46:14.529448
3,EDDB,EVRA,BT 211,airBaltic,2026-03-03 07:55:00,1,Riga,2026-03-03 14:46:14.529458
4,EDDB,EVRA,JU 7659,JU,2026-03-03 07:55:00,1,Riga,2026-03-03 14:46:14.529471


In [78]:
flights_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 109 entries, 0 to 108
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   arrival_aeroport_icao   109 non-null    str           
 1   departure_airport_icao  109 non-null    str           
 2   flight_number           109 non-null    str           
 3   airline                 109 non-null    str           
 4   arrival_time            109 non-null    datetime64[us]
 5   arrival_terminal        108 non-null    str           
 6   departure_city          109 non-null    str           
 7   data_retrieved_on       109 non-null    datetime64[us]
dtypes: datetime64[us](2), str(6)
memory usage: 6.9 KB


While string slicing provides a quick solution to correcting the `scheduled_arrival_time` column, it's not the most robust approach. This is because it assumes that every cell in the column has the `+01:00` time zone offset. If there are cells without this offset, slicing would remove part of the time value, leading to inaccurate results.

A more robust solution would involve using the `re.sub()` function from the re module. 

In [79]:
#alternative:
import re

# the pattern looks for + or -, two digits, a colon, and two digits at the end ($)
pattern = r'[+-]\d{2}:\d{2}$'

# clean the column
flights_df["arrival_time"] = flights_df["arrival_time"].apply(
    lambda x: re.sub(pattern, '', str(x))
)

# convert to datetime so we can actually use the data
flights_df["arrival_time"] = pd.to_datetime(flights_df["arrival_time"])

print(flights_df["arrival_time"].head(1))
# output: 2026-03-03 11:50:00


0   2026-03-03 07:55:00
Name: arrival_time, dtype: datetime64[us]


4.3. Create function to collect flights info in different cities

In [80]:
import requests
import pandas as pd
from sqlalchemy import create_engine
from datetime import datetime, timedelta
import time
import os

def get_flights(connection_string, API_key):
    """
    Fetch flights for airports listed in MySQL's 'airports' and 'cities_airports' tables,
    process the data, and insert the cleaned data into the 'flights' table.
    """

    # fetch all ICAO codes from the 'airports' table
    engine = create_engine(connection_string)
    airports_df = pd.read_sql("SELECT airport_icao FROM airports", engine)

    # construct the time window for tomorrow's arrivals
    tomorrow = (datetime.now() + timedelta(days=1)).date()
    # split the day into two 12-hour chunks to avoid the 400 error
    time_windows = [
        (f"{tomorrow}T00:00", f"{tomorrow}T11:59"),
        (f"{tomorrow}T12:00", f"{tomorrow}T23:59")
    ]

    # setup headers for the AeroDataBox API
    headers = {
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "x-rapidapi-key": API_key
    }

    all_flight_items = []

    # loop through each ICAO code in the airports table
    for icao in airports_df['airport_icao']:
        for start, end in time_windows:
        # fetch flight data for the airport
            url_flights = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{start}/{end}"
            querystring = {
            "withLeg": "true",
            "direction": "Arrival",
            "withCancelled": "false",
            "withCodeshared": "true",
            "withCargo": "false",
            "withPrivate": "false"
            }

        print(f"Requesting {icao} from {start} to {end}...")
        response_flights = requests.get(url_flights, headers=headers, params=querystring)

        if response_flights.status_code == 200:
            flights_json = response_flights.json()

            if "arrivals" in flights_json:
                for item in flights_json["arrivals"]:
                    flight_item = {
                        "arrival_aeroport_icao": icao,  # arrival airport ICAO (from the loop or API)
                        "departure_airport_icao": item["departure"]["airport"].get("icao", None),  # departure airport ICAO
                        "flight_number": item.get("number", None),  # flight number
                        "airline": item.get("airline", {}).get("name", None),  # airline name (United)
                        "arrival_time": item.get("arrival", {}).get("scheduledTime", {}).get("local", None),  # scheduled arrival time
                        "arrival_terminal": item.get("arrival", {}).get("terminal", None),  # arrival terminal
                        "departure_city": item.get("departure", {}).get("airport", {}).get("name", None),  # departure city (airport name)
                        "data_retrieved_on": datetime.now()  # timestamp of when the data was retrieved
                    }
                    all_flight_items.append(flight_item)
        else:
            print(f"Failed {icao} ({start}): {response.status_code} - {response.text}")

        # sleep to avoid hitting the API rate limit
        time.sleep(1.5)

    # if no flight data is found, return an empty DataFrame
    if not all_flight_items:
        print("No flight data found.")
        return pd.DataFrame()

    # create a DataFrame from the fetched flight data
    flights_df = pd.DataFrame(all_flight_items)

    import re
    flights_df["arrival_time"] = flights_df["arrival_time"].apply(
        lambda x: re.sub(r'[+-]\d{2}:\d{2}$', '', str(x)) if x else None
    )

    # clean and convert the arrival time to a datetime object
    flights_df["arrival_time"] = pd.to_datetime(flights_df["arrival_time"])
    flights_df["data_retrieved_on"] = pd.to_datetime(flights_df["data_retrieved_on"])

    return flights_df

In [81]:
# configuration and connection
def connection_string_local():
    schema = "sql_gans"
    host = "127.0.0.1"
    user = "root"
    password = os.getenv("password")
    port = 3306

    if password is None:
        raise ValueError("ERROR: 'password' not found in .env file. Check your filename and path.")

# build connection string only if password exists
    connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
    print("Connection string created successfully.")
    return connection_string

In [82]:
connection_string = connection_string_local()

Connection string created successfully.


In [ ]:
# call the function
API_key = os.getenv("AeroDataBoxApi")
final_flights_df = get_flights(connection_string, API_key)
final_flights_df

Requesting EDDB from 2026-03-04T12:00 to 2026-03-04T23:59...
Requesting EDDH from 2026-03-04T12:00 to 2026-03-04T23:59...
Requesting EDDM from 2026-03-04T12:00 to 2026-03-04T23:59...


,arrival_aeroport_icao,departure_airport_icao,flight_number,airline,arrival_time,arrival_terminal,departure_city,data_retrieved_on
0,EDDB,LPPT,TP 530,TAP Air Portugal,2026-03-04 12:00:00,1,Lisbon,2026-03-03 14:47:01.405078
1,EDDB,LTFM,TK 1729,Turkish,2026-03-04 12:05:00,1,Istanbul,2026-03-03 14:47:01.405105
2,EDDB,EDDM,LH 2206,Lufthansa,2026-03-04 12:10:00,1,Munich,2026-03-03 14:47:01.405111
3,EDDB,EDDM,OU 5360,Croatia,2026-03-04 12:10:00,1,Munich,2026-03-03 14:47:01.405119
4,EDDB,EDDM,AC 9049,Air Canada,2026-03-04 12:10:00,1,Munich,2026-03-03 14:47:01.405125
...,...,...,...,...,...,...,...,...
588,EDDM,LCLK,4Y 1285,Flight Alaska,2026-03-04 23:00:00,2,Larnarca,2026-03-03 14:47:06.562567
589,EDDM,LPPT,TP 556,TAP Air Portugal,2026-03-04 23:05:00,2,Lisbon,2026-03-03 14:47:06.562574
590,EDDM,EGLL,VL 2483,Lufthansa City,2026-03-04 23:05:00,2,London,2026-03-03 14:47:06.562581
591,EDDM,EDDF,LH 124,Lufthansa,2026-03-04 23:10:00,2,Frankfurt-am-Main,2026-03-03 14:47:06.562589


4.4. Send our flights DataFrame to MYSQL

In [ ]:
from sqlalchemy import create_engine

def send_flights_to_sql(final_flights_df, connection_string):
    """
    Inserts data into the 'flights' table.
    Matches your exact CREATE TABLE schema.
    """
    if final_flights_df.empty:
        print("No data to upload.")
        return

    # list only the columns we want to send
    # do not include 'flight_id' - MySQL creates it automatically
    sql_columns = [
        "arrival_aeroport_icao",
        "flight_number",
        "airline",
        "arrival_time",
        "arrival_terminal",
        "departure_city",
        "departure_airport_icao",
        "data_retrieved_on"
    ]

    try:
        engine = create_engine(connection_string)

        # filter the DataFrame to only include the columns in the SQL table
        # this prevents "Unknown Column" errors
        df_to_insert = final_flights_df[sql_columns].copy()

        # insert into MySQL
        df_to_insert.to_sql(
            name='flights',
            con=engine,
            if_exists='append',
            index=False
        )
        print(f"Successfully uploaded {len(df_to_insert)} rows to 'flights'.")

    except Exception as e:
        # this will print the EXACT reason for the failure (e.g., Foreign Key error)
        print(f"SQL Error details: {e}")

# call the function
send_flights_to_sql(final_flights_df, connection_string)



Successfully uploaded 593 rows to 'flights'.


4.5. Create a function  from grabbing the data from airports table to send flights data to MySQL

In [ ]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta
import time
import os
import re

# function to create the connection string
def connection_string_local():
    schema = "sql_gans"
    host = "127.0.0.1"
    user = "root"
    password = os.getenv("password")
    port = 3306

    if password is None:
        raise ValueError("ERROR: 'password' not found in environment variables.")

    connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
    print("Connection string created successfully.")
    return connection_string

# function to fetch and clean flight data
def get_flights(connection_string, API_key):
    engine = create_engine(connection_string)
    airports_df = pd.read_sql("SELECT airport_icao FROM airports", engine)

    tomorrow = (datetime.now() + timedelta(days=1)).date()
    time_windows = [
        (f"{tomorrow}T00:00", f"{tomorrow}T11:59"),
        (f"{tomorrow}T12:00", f"{tomorrow}T23:59")
    ]

    headers = {
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "x-rapidapi-key": API_key
    }

    all_flight_items = []

    for icao in airports_df['airport_icao']:
        for start, end in time_windows:
            url_flights = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{start}/{end}"
            querystring = {
            "withLeg": "true",
            "direction": "Arrival",
            "withCancelled": "false",
            "withCodeshared": "true",
            "withCargo": "false",
            "withPrivate": "false"
            }
            print(f"Requesting {icao} from {start} to {end}...")
            response = requests.get(url_flights, headers=headers, params=querystring)

            if response.status_code == 200:
                flights_json = response.json()
                if "arrivals" in flights_json:
                    for item in flights_json["arrivals"]:
                        all_flight_items.append({
                            "arrival_aeroport_icao": icao,
                            "departure_airport_icao": item.get("departure", {}).get("airport", {}).get("icao"),
                            "flight_number": item.get("number"),
                            "airline": item.get("airline", {}).get("name"),
                            "arrival_time": item.get("arrival", {}).get("scheduledTime", {}).get("local"),
                            "arrival_terminal": item.get("arrival", {}).get("terminal"),
                            "departure_city": item.get("departure", {}).get("airport", {}).get("name"),
                            "data_retrieved_on": datetime.now()
                        })
            else:
                print(f"Failed {icao}: {response.status_code} - {response.text}")

            time.sleep(1.5)

    if not all_flight_items:
        return pd.DataFrame()

    flights_df = pd.DataFrame(all_flight_items)

    # robust cleaning of timezone offset
    flights_df["arrival_time"] = flights_df["arrival_time"].apply(
        lambda x: re.sub(r'[+-]\d{2}:\d{2}$', '', str(x)) if x else None
    )

    # clean and convert the arrival time to a datetime object
    flights_df["arrival_time"] = pd.to_datetime(flights_df["arrival_time"])
    flights_df["data_retrieved_on"] = pd.to_datetime(flights_df["data_retrieved_on"])

    return flights_df

# function to insert data into SQL
def send_flights_to_sql(final_flights_df, connection_string):
    if final_flights_df.empty:
        print("No data to upload.")
        return

    # list only columns matching our SQL table (EXCLUDING flight_id)
    sql_columns = [
        "arrival_aeroport_icao",
        "flight_number",
        "airline",
        "arrival_time",
        "arrival_terminal",
        "departure_city",
        "departure_airport_icao",
        "data_retrieved_on"
    ]

    try:
        engine = create_engine(connection_string)
        df_to_insert = final_flights_df[sql_columns].copy()

        df_to_insert.to_sql(
            name='flights',
            con=engine,
            if_exists='append',
            index=False
        )
        print(f"Successfully uploaded {len(df_to_insert)} rows.")
    except Exception as e:
        print(f"SQL Error: {e}")

# --- EXECUTION FLOW ---
# A. Build connection
connection_string = connection_string_local()

# B. Get API key
API_key = os.getenv("AeroDataBoxApi")

# C. Run Fetching
final_flights_df = get_flights(connection_string, API_key)

# D. Run Upload
send_flights_to_sql(final_flights_df, connection_string)


Connection string created successfully.
Requesting EDDB from 2026-03-04T00:00 to 2026-03-04T11:59...
Requesting EDDB from 2026-03-04T12:00 to 2026-03-04T23:59...
Requesting EDDH from 2026-03-04T00:00 to 2026-03-04T11:59...
Requesting EDDH from 2026-03-04T12:00 to 2026-03-04T23:59...
Requesting EDDM from 2026-03-04T00:00 to 2026-03-04T11:59...
Requesting EDDM from 2026-03-04T12:00 to 2026-03-04T23:59...
Successfully uploaded 874 rows.


4.6. Create a function with lists of ICAO to collect flights data 

In [ ]:
import requests
import pandas as pd
#from sqlalchemy import create_engine, text
from datetime import datetime, timedelta
import time
import os
import re

# function to fetch and clean flight data
def get_all_cities_flights(icao_list):
     # API Configuration
    API_key = os.getenv("AeroDataBoxApi")
    headers = {
        'x-rapidapi-host': "aerodatabox.p.rapidapi.com",
        'x-rapidapi-key': API_key
    }
    # time Window (AeroDataBox 12-hour limit)
    tomorrow = (datetime.now() + timedelta(days=1)).date()
    time_windows = [
        (f"{tomorrow}T00:00", f"{tomorrow}T11:59"),
        (f"{tomorrow}T12:00", f"{tomorrow}T23:59")
    ]

    all_flight_items = []

    for icao in icao_list:
        for start, end in time_windows:
            url_flights = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{start}/{end}"
            querystring = {
            "withLeg": "true",
            "direction": "Arrival",
            "withCancelled": "false",
            "withCodeshared": "true",
            "withCargo": "false",
            "withPrivate": "false"
            }
            print(f"Requesting {icao} from {start} to {end}...")
            response = requests.get(url_flights, headers=headers, params=querystring)

            if response.status_code == 200:
                flights_json = response.json()
                if "arrivals" in flights_json:
                    for item in flights_json["arrivals"]:
                        all_flight_items.append({
                            "arrival_aeroport_icao": icao,
                            "departure_airport_icao": item.get("departure", {}).get("airport", {}).get("icao"),
                            "flight_number": item.get("number"),
                            "airline": item.get("airline", {}).get("name"),
                            "arrival_time": item.get("arrival", {}).get("scheduledTime", {}).get("local"),
                            "arrival_terminal": item.get("arrival", {}).get("terminal"),
                            "departure_city": item.get("departure", {}).get("airport", {}).get("name"),
                            "data_retrieved_on": datetime.now().strftime("%Y-%m-%d %H:%M")
                        })
            else:
                print(f"Failed {icao}: {response.status_code} - {response.text}")

            time.sleep(1.5)

    if not all_flight_items:
        return pd.DataFrame()

    flights_df = pd.DataFrame(all_flight_items)

    # robust cleaning of timezone offset
    flights_df["arrival_time"] = flights_df["arrival_time"].apply(
        lambda x: re.sub(r'[+-]\d{2}:\d{2}$', '', str(x)) if x else None
    )

    # clean and convert the arrival time to a datetime object
    flights_df["arrival_time"] = pd.to_datetime(flights_df["arrival_time"])
    flights_df["data_retrieved_on"] = pd.to_datetime(flights_df["data_retrieved_on"])

    return flights_df

In [92]:
# call the function
icao_list = ["EDDB", "EDDF"]

# run the function and get the final DataFrame
flights_df = get_all_cities_flights(icao_list)
flights_df

Requesting EDDB from 2026-03-04T00:00 to 2026-03-04T11:59...
Requesting EDDB from 2026-03-04T12:00 to 2026-03-04T23:59...
Requesting EDDF from 2026-03-04T00:00 to 2026-03-04T11:59...
Requesting EDDF from 2026-03-04T12:00 to 2026-03-04T23:59...


,arrival_aeroport_icao,departure_airport_icao,flight_number,airline,arrival_time,arrival_terminal,departure_city,data_retrieved_on
0,EDDB,ZBAA,HU 489,Hainan,2026-03-04 06:40:00,1,Beijing,2026-03-03 15:02:00
1,EDDB,EDDF,LH 172,Lufthansa,2026-03-04 07:50:00,1,Frankfurt-am-Main,2026-03-03 15:02:00
2,EDDB,EDDF,VL 172,Lufthansa City,2026-03-04 07:50:00,1,Frankfurt-am-Main,2026-03-03 15:02:00
3,EDDB,ELLX,LG 9471,Luxair,2026-03-04 07:50:00,1,Luxembourg,2026-03-03 15:02:00
4,EDDB,KEWR,UA 962,United,2026-03-04 07:55:00,1,Newark,2026-03-03 15:02:00
...,...,...,...,...,...,...,...,...
2271,EDDF,LPPT,TP 574,TAP Air Portugal,2026-03-04 22:40:00,1,Lisbon,2026-03-03 15:02:00
2272,EDDF,LPPT,AC 2660,Air Canada,2026-03-04 22:40:00,1,Lisbon,2026-03-03 15:02:00
2273,EDDF,LPPT,S4 8762,Azores,2026-03-04 22:40:00,1,Lisbon,2026-03-03 15:02:00
2274,EDDF,LPPT,LH 6957,Lufthansa,2026-03-04 22:40:00,1,Lisbon,2026-03-03 15:02:00
